# Nuage de points sur carte : éditions de Venise, Paris et Lyon

Pour ces trois villes d'édition des *Métamorphoses* d'Ovide, un nuage de points sur une
**vraie carte géographique historique** : Venise, Paris et Lyon sont à leur
position réelle, et autour de chaque ville un petit nuage de points en spirale, un point par
**éditeur** ayant publié dans cette ville (taille ∝ nombre d'éditions publiées). Avec un tableau
dépliable liste tout pour chaque ville.

Même source de données que `01_carte_circulation.ipynb` : `retours_celine/BNU_corpus.ods`
(feuille `Synthèse`).

In [7]:
import os
import re
import json
from odf.opendocument import load as charger_ods
from odf.table import Table, TableRow, TableCell
from odf.text import P
from odf import teletype

RACINE = os.path.abspath("../..")
DOSSIER_VIZ = os.path.join(RACINE, "resultats", "Datavis")
os.makedirs(DOSSIER_VIZ, exist_ok=True)
CHEMIN_SORTIE = os.path.join(DOSSIER_VIZ, "nuage_editions_venise_paris_lyon.html")
CHEMIN_CORPUS = os.path.join(RACINE, "retours_celine", "BNU_corpus.ods")

## 1. Chargement du corpus

Même lecteur ODS cellule par cellule que pour la carte (`pandas.read_excel` masque des
colonnes de ce fichier — voir `01_carte_circulation.ipynb` pour le détail). On ne garde que les
éditions dont la ville normalisée est Lyon, Paris ou Venise.

In [ ]:
def lire_feuille_ods(chemin, nom_feuille):
    """Lit une feuille ODS cellule par cellule (pandas ignore des colonnes de ce fichier)."""
    doc = charger_ods(chemin)
    table = next(t for t in doc.spreadsheet.getElementsByType(Table)
                 if t.getAttribute("name") == nom_feuille)
    lignes_brutes = table.getElementsByType(TableRow)

    def valeurs_ligne(ligne):
        valeurs, col = {}, 0
        for cellule in ligne.getElementsByType(TableCell):
            rep = cellule.getAttribute("numbercolumnsrepeated")
            rep = int(rep) if rep else 1
            paras = cellule.getElementsByType(P)
            texte = " ".join(teletype.extractText(p) for p in paras)
            for k in range(rep):
                valeurs[col + k] = texte
            col += rep
        return valeurs

    entetes = valeurs_ligne(lignes_brutes[0])
    colonnes = {i: t.strip() for i, t in entetes.items() if t.strip()}

    lignes = []
    for ligne in lignes_brutes[1:]:
        rep = ligne.getAttribute("numberrowsrepeated")
        rep = int(rep) if rep else 1
        valeurs = valeurs_ligne(ligne)
        if not any(v.strip() for v in valeurs.values()):
            continue  # ligne vide (fin de feuille)
        d = {nom: valeurs.get(i, "").strip() for i, nom in colonnes.items()}
        lignes.extend([d] * rep)
    return lignes

def extraire_annee(valeur):
    """Renvoie la première année à 4 chiffres trouvée (ex: "1527 / 1528 ?" -> 1527)."""
    m = re.search(r"\d{4}", str(valeur))
    return int(m.group()) if m else None

def extraire_lien(row):
    """Choisit le premier lien exploitable, par ordre de préférence (même logique que
    01_carte_circulation.ipynb) : certaines cellules contiennent plusieurs liens séparés par
    ';' (ex. plusieurs notices catalogue) — on ne garde que le premier, sinon le lien final
    serait une concaténation invalide (ex. Lyon 1516, Biblioteca Digital Ovidiana contenait
    une seule URL mais suivie d'un ';?%3E' résiduel qui aurait été inclus tel quel)."""
    for col in ["version numérisée 1", "version numérisée 2", "Biblioteca Digital Ovidiana", "url catalogue"]:
        val = row.get(col, "")
        if not val:
            continue
        premier = val.split(";")[0].strip()
        if premier.startswith("http"):
            return premier
    return None

def graveur_ou_inconnu(g):
    """Repli neutre pour un graveur non identifié ('?', 'inaccessible', case vide) — les
    'AnonymeXXXX' (X = année) sont eux déjà des identifiants distincts d'un graveur à l'autre,
    contrairement à 's.n.' pour les éditeurs (voir plus bas) : pas besoin de les numéroter."""
    g = (g or "").strip()
    if not g or g.lower() in {"?", "inaccessible"}:
        return "Graveur non identifié"
    return g

# Seules les 3 villes qui nous intéressent ici 
VILLES_RETENUES = {"Paris": "Paris", "[Paris]": "Paris", "Lyon": "Lyon", "Venise": "Venise"}
ORDRE_VILLES = ["Lyon", "Paris", "Venise"]

COL_GRAVEUR = "graveur\xa0: Nom, Prénom"

corpus = lire_feuille_ods(CHEMIN_CORPUS, "Synthèse")

def fusionner_tomes(corpus):
    """Une même édition est parfois scindée en plusieurs tomes dans le tableau source (une
    ligne par tome : même ville/année/éditeur/titre abrégé/graveur, seul le titre complet
    varie selon le tome — ex. Lyon 1697 "Les Oeuvres d'Ovide..." en 3 tomes, Amsterdam 1693 en
    3 tomes). On les fusionne en une seule édition : sinon elles compteraient 2 ou 3 fois pour
    ce qui est en réalité une seule publication. Le lien et le commentaire de copie, souvent
    renseignés sur un seul des tomes, sont récupérés du premier tome qui les a."""
    groupes, ordre = {}, []
    for ligne in corpus:
        cle = (ligne.get("ville", ""), ligne.get("année", ""), ligne.get("publisher", ""),
               ligne.get("titre abrégé", ""), ligne.get(COL_GRAVEUR, ""))
        if cle not in groupes:
            groupes[cle] = []
            ordre.append(cle)
        groupes[cle].append(ligne)

    champs_premier_non_vide = ["url catalogue", "version numérisée 1", "version numérisée 2",
                                "Biblioteca Digital Ovidiana", "copies de cette édition"]
    fusionne = []
    for cle in ordre:
        lignes_tomes = groupes[cle]
        base = dict(lignes_tomes[0])
        if len(lignes_tomes) > 1:
            for champ in champs_premier_non_vide:
                for ligne in lignes_tomes:
                    if ligne.get(champ, "").strip():
                        base[champ] = ligne[champ]
                        break
        fusionne.append(base)
    return fusionne

nb_avant_fusion = len(corpus)
corpus = fusionner_tomes(corpus)
if len(corpus) != nb_avant_fusion:
    print(nb_avant_fusion - len(corpus), "lignes fusionnées (tomes d'une même édition regroupés)")

editions = []
for row in corpus:
    ville = VILLES_RETENUES.get(row.get("ville", "").strip())
    if ville is None:
        continue
    annee = extraire_annee(row.get("année", ""))
    if annee is None:
        continue
    titre = row.get("titre abrégé") or row.get("titre complet") or ""
    editions.append({
        "ville": ville,
        "annee": annee,
        "titre": titre.strip(),
        "graveur": graveur_ou_inconnu(row.get(COL_GRAVEUR, "")),
        "publisher": row.get("publisher", "").strip() or "Éditeur non identifié",
        "lien": extraire_lien(row),
    })

# "s.n." ("sine nomine" : éditeur non mentionné sur l'édition) n'est pas un nom d'éditeur —
# plusieurs éditions "s.n." dans une même ville ne sont pas forcément du même éditeur. Sans
# ce correctif, elles seraient regroupées à tort en un seul point sur la carte (comme si
# c'était un unique éditeur très actif). On les distingue donc par un numéro (s.n.1, s.n.2...),
# uniquement quand il y en a plus d'une dans la ville (sinon le numéro n'apporte rien).
for ville in ORDRE_VILLES:
    inconnus = [e for e in editions if e["ville"] == ville and e["publisher"].strip().lower() == "s.n."]
    if len(inconnus) > 1:
        for i, e in enumerate(sorted(inconnus, key=lambda e: e["annee"]), start=1):
            e["publisher"] = f"s.n.{i}"

print(len(editions), "éditions retenues (Lyon, Paris, Venise)")
for v in ORDRE_VILLES:
    sous = [e for e in editions if e["ville"] == v]
    print(f"  {v:8s} {len(sous):2d} éditions, {min(e['annee'] for e in sous)}–{max(e['annee'] for e in sous)}, "
          f"{len({e['publisher'] for e in sous})} éditeurs distincts, "
          f"{len({e['graveur'] for e in sous})} graveurs distincts")

## 2. Choix de visualisation


In [ ]:
import math

# Mêmes coordonnées que 01_carte_circulation.ipynb
VILLES_COORDS = {"Lyon": (45.764, 4.8357), "Paris": (48.8566, 2.3522), "Venise": (45.4408, 12.3155)}

RAYON_MAX_KM = 22            # étalement maximal du nuage d'éditeurs autour de chaque ville
ANGLE_OR = math.radians(137.508)  # angle d'or : répartition régulière en spirale, sans grille

def decalage_spirale(i, n, rayon_max_km):
    """Spirale de phyllotaxie (rayon ∝ √i, angle = i × angle d'or) : répartit n'importe quel
    nombre de points régulièrement autour d'un centre, sans qu'ils se chevauchent."""
    if n <= 1:
        return 0.0, 0.0
    rayon = rayon_max_km * math.sqrt(i / (n - 1))
    angle = i * ANGLE_OR
    return rayon * math.cos(angle), rayon * math.sin(angle)

def deplacer_latlon(lat, lon, dx_km, dy_km):
    """Déplace un point de (dx_km vers l'est, dy_km vers le nord) en degrés lat/lon."""
    dlat = dy_km / 111.0
    dlon = dx_km / (111.0 * math.cos(math.radians(lat)))
    return lat + dlat, lon + dlon

# Une entrée par édition, pour le tableau détaillé de chaque ville (section 3).
points = [
    {
        "ville": e["ville"],
        "annee": e["annee"],
        "titre": e["titre"],
        "graveur": e["graveur"],
        "editeur": e["publisher"],
        "lien": e["lien"],
    }
    for e in editions
]

print(len(points), "éditions prêtes pour les tableaux détaillés")

In [ ]:
import colorsys

def couleur_categorielle(i, saturation=0.55, luminosite=0.48):
    """Couleur hex distincte pour l'index i, par rotation à l'angle d'or (voir ANGLE_OR
    plus haut) : n'importe quel nombre de catégories reste bien réparti sur le cercle
    chromatique, sans avoir à choisir une palette manuelle à l'avance."""
    teinte = (i * 137.508 % 360) / 360
    r, g, b = colorsys.hls_to_rgb(teinte, luminosite, saturation)
    return "#{:02x}{:02x}{:02x}".format(round(r * 255), round(g * 255), round(b * 255))

groupes_editeurs = {}
for e in editions:
    groupes_editeurs.setdefault((e["ville"], e["publisher"]), []).append(e)

points_editeurs = []
for ville in VILLES_COORDS:
    editeurs_ville = sorted(
        {cle[1] for cle in groupes_editeurs if cle[0] == ville},
        key=lambda editeur: (-len(groupes_editeurs[(ville, editeur)]), editeur)
    )
    n = len(editeurs_ville)
    lat0, lon0 = VILLES_COORDS[ville]
    for i, editeur in enumerate(editeurs_ville):
        couleur = couleur_categorielle(i)
        eds = sorted(groupes_editeurs[(ville, editeur)], key=lambda e: e["annee"])
        dx, dy = decalage_spirale(i, n, RAYON_MAX_KM)
        lat, lon = deplacer_latlon(lat0, lon0, dx, dy)
        points_editeurs.append({
            "lat": round(lat, 5),
            "lon": round(lon, 5),
            "ville": ville,
            "editeur": editeur,
            "couleur": couleur,
            "editions": [
                {"annee": e["annee"], "titre": e["titre"], "graveur": e["graveur"], "lien": e["lien"]}
                for e in eds
            ],
        })

print(len(points_editeurs), "éditeurs positionnés sur la carte (taille ∝ nombre d'éditions, couleur propre à chacun)")
for ville in VILLES_COORDS:
    sous = [p for p in points_editeurs if p["ville"] == ville]
    plus_actif = max(sous, key=lambda p: len(p["editions"]))
    print(f"  {ville:8s} {len(sous):2d} éditeurs — le plus actif : {plus_actif['editeur']} "
          f"({len(plus_actif['editions'])} éditions)")

In [ ]:
# Même logique que points_editeurs, mais groupée par graveur plutôt que par éditeur.
groupes_graveurs = {}
for e in editions:
    groupes_graveurs.setdefault((e["ville"], e["graveur"]), []).append(e)

points_graveurs = []
for ville in VILLES_COORDS:
    graveurs_ville = sorted(
        {cle[1] for cle in groupes_graveurs if cle[0] == ville},
        key=lambda graveur: (-len(groupes_graveurs[(ville, graveur)]), graveur)
    )
    n = len(graveurs_ville)
    lat0, lon0 = VILLES_COORDS[ville]
    for i, graveur in enumerate(graveurs_ville):
        couleur = couleur_categorielle(i)
        eds = sorted(groupes_graveurs[(ville, graveur)], key=lambda e: e["annee"])
        dx, dy = decalage_spirale(i, n, RAYON_MAX_KM)
        lat, lon = deplacer_latlon(lat0, lon0, dx, dy)
        points_graveurs.append({
            "lat": round(lat, 5),
            "lon": round(lon, 5),
            "ville": ville,
            "graveur": graveur,
            "couleur": couleur,
            "editions": [
                {"annee": e["annee"], "titre": e["titre"], "editeur": e["publisher"], "lien": e["lien"]}
                for e in eds
            ],
        })

print(len(points_graveurs), "graveurs positionnés sur la carte (taille ∝ nombre d'éditions, couleur propre à chacun)")
for ville in VILLES_COORDS:
    sous = [p for p in points_graveurs if p["ville"] == ville]
    plus_actif = max(sous, key=lambda p: len(p["editions"]))
    print(f"  {ville:8s} {len(sous):2d} graveurs — le plus actif : {plus_actif['graveur']} "
          f"({len(plus_actif['editions'])} éditions)")

## 3. Génération de la carte (HTML autonome)

In [ ]:
def ligne_tableau(p):
    lien_html = f'<a href="{p["lien"]}" target="_blank">voir</a>' if p["lien"] else ""
    return (
        f'<tr><td>{p["annee"]}</td><td>{p["titre"]}</td>'
        f'<td>{p["editeur"]}</td><td>{p["graveur"]}</td>'
        f'<td>{lien_html}</td></tr>'
    )

def tableau_ville(ville):
    lignes = "\n".join(
        ligne_tableau(p) for p in sorted(points, key=lambda p: p["annee"]) if p["ville"] == ville
    )
    return f'''<table class="tableau-detaille" id="tableau-{ville}">
    <thead><tr><th>Année</th><th>Titre</th><th>Éditeur</th><th>Graveur</th><th>Lien</th></tr></thead>
    <tbody>
      {lignes}
    </tbody>
  </table>'''

# Une section par ville : carte, bouton pour déplier le tableau détaillé, puis le tableau.
cellules = []
for ville in VILLES_COORDS:
    cellules.append(f'''<div class="cellule-carte">
    <h2>{ville}</h2>
    <div class="carte" id="carte-{ville}"></div>
    <button class="bascule action-ville" data-ville="{ville}">Afficher le tableau détaillé</button>
    {tableau_ville(ville)}
  </div>''')
blocs_carte = "\n".join(cellules)

TEMPLATE_HTML = r"""<!DOCTYPE html>
<html lang="fr"><head><meta charset="utf-8">
<title>Nuage de points : éditions de Venise, Paris et Lyon</title>
<link rel="stylesheet" href="https://unpkg.com/leaflet@1.9.4/dist/leaflet.css"/>
<script src="https://unpkg.com/leaflet@1.9.4/dist/leaflet.js"></script>
<link href="https://unpkg.com/maplibre-gl@3.6.2/dist/maplibre-gl.css" rel="stylesheet"/>
<script src="https://unpkg.com/maplibre-gl@3.6.2/dist/maplibre-gl.js"></script>
<script src="https://unpkg.com/@maplibre/maplibre-gl-leaflet@0.0.20/leaflet-maplibre-gl.js"></script>
<style>
  :root {
    --surface: #fffaf0; --texte-fort: #2b1e15; --texte-att: #6b5c4f; --trait: #d8cfc0;
    --contour-point: #3e2c23; --c-lien: #2a78d6;
  }
  @media (prefers-color-scheme: dark) {
    :root {
      --surface: #1a1a19; --texte-fort: #f2ece2; --texte-att: #c3baa9; --trait: #3a352c;
      --contour-point: #f2ece2; --c-lien: #3987e5;
    }
  }
  body { margin:0; font-family:Georgia,serif; background:var(--surface); color:var(--texte-fort); }
  .page { max-width:1100px; margin:0 auto; padding:16px 20px 32px; position:relative; }
  h1 { font-size:19px; margin:0 0 4px; }
  p.souschapo { font-size:13px; color:var(--texte-att); margin:0 0 10px; }

  .commandes-globales { display:flex; align-items:center; gap:8px; font-size:12px;
    color:var(--texte-att); margin:0 0 14px; }
  .bascule-groupement { font-family:Georgia,serif; font-size:12px; background:none;
    border:1px solid var(--trait); color:var(--texte-fort); border-radius:4px; padding:5px 10px;
    cursor:pointer; }
  .bascule-groupement.actif { background:var(--texte-fort); color:var(--surface); border-color:var(--texte-fort); }

  /* Les 3 cartes empilées verticalement, une par ville. */
  .rangee-cartes { display:flex; flex-direction:column; gap:20px; margin-top:14px; }
  .cellule-carte h2 { font-size:15px; margin:0 0 6px; color:var(--texte-fort); }
  .carte { height:380px; border-radius:6px; box-shadow:0 1px 6px rgba(0,0,0,.25); }

  .action-ville { display:block; margin:8px 0 0; }
  button.bascule { font-family:Georgia,serif; font-size:12px; background:none;
    border:1px solid var(--trait); color:var(--texte-fort); border-radius:4px; padding:5px 10px;
    cursor:pointer; }
  table.tableau-detaille { width:100%; border-collapse:collapse;
    font-size:12px; margin:6px 0 0; display:none; }
  table.tableau-detaille.visible { display:table; }
  table.tableau-detaille th, table.tableau-detaille td { text-align:left; padding:4px 8px;
    border-bottom:1px solid var(--trait); }
  table.tableau-detaille a { color:var(--c-lien); }

  .leaflet-tooltip { font-family:Georgia,serif; font-size:12px; background:var(--surface);
    border:1px solid var(--contour-point); color:var(--texte-fort); padding:4px 9px;
    box-shadow:0 1px 5px rgba(0,0,0,.3); }
  /* Popups Leaflet (clic sur un éditeur ou un graveur) : même thème parchemin, contenu
     défilable si la liste d'éditions est longue, liens "voir" bien cliquables (contrairement
     à une infobulle au survol, une popup reste ouverte tant qu'on ne clique pas ailleurs). */
  .leaflet-popup-content-wrapper { background:var(--surface); color:var(--texte-fort);
    border:1px solid var(--contour-point); border-radius:6px; }
  .leaflet-popup-tip { background:var(--surface); border:1px solid var(--contour-point); }
  .leaflet-popup-content { font-family:Georgia,serif; font-size:12px; max-height:220px;
    overflow-y:auto; margin:10px 12px; }
  .leaflet-popup-content hr { border:none; border-top:1px solid var(--trait); margin:4px 0; }
  .ed-popup { padding:3px 0; border-bottom:1px solid var(--trait); }
  .ed-popup:last-child { border-bottom:none; }
  .ed-popup a { color:var(--c-lien); }
</style></head><body>
<div class="page">
  <h1>Éditions de Venise, Paris et Lyon</h1>
  <p class="souschapo">Chaque point regroupe les éditions d'un même éditeur (ou graveur) : sa
    taille indique combien. Survolez pour un aperçu, cliquez pour le détail de chaque édition
    et son lien vers la version numérisée.</p>
  <div class="commandes-globales">
    <span>Regrouper les points de la carte par :</span>
    <button class="bascule-groupement actif" data-mode="editeur">Éditeur</button>
    <button class="bascule-groupement" data-mode="graveur">Graveur</button>
  </div>
  <div class="rangee-cartes">
    __BLOCS_CARTE__
  </div>
</div>
<script>
  const pointsEditeurs = __POINTS_EDITEURS__;
  const pointsGraveurs = __POINTS_GRAVEURS__;
  const villesCoords = __VILLES_COORDS__;

  // Construit les marqueurs Leaflet d'un jeu de points (éditeurs ou graveurs) pour une ville.
  // `cle` vaut 'editeur' ou 'graveur' : nom de la clé qui porte le nom du groupe dans `p`.
  // Chaque édition listée dans la popup affiche en plus l'autre role (le graveur quand les
  // points sont des éditeurs, l'éditeur quand les points sont des graveurs) : le nom du
  // groupe lui-même est déjà dans l'entête de la popup, mais pas cette autre information,
  // qui elle varie d'une édition à l'autre au sein d'un même point.
  function construireMarqueurs(map, liste, ville, cle) {
    const autreCle = cle === 'editeur' ? 'graveur' : 'editeur';
    return liste.filter(p => p.ville === ville).map(p => {
      const n = p.editions.length;
      const rayon = 6 + Math.sqrt(n) * 5;
      const nom = p[cle];
      const editionsTriees = p.editions.slice().sort((a, b) => a.annee - b.annee);
      const listeEditions = editionsTriees.map(e =>
        '<div class="ed-popup"><b>' + e.annee + '</b> — ' + e.titre +
        ' <i>(' + e[autreCle] + ')</i>' +
        (e.lien ? ' <a href="' + e.lien + '" target="_blank">→ voir</a>' : '') + '</div>'
      ).join('');
      return L.circleMarker([p.lat, p.lon], {
        radius: rayon, color:'#3e2c23', weight:1, fillColor: p.couleur, fillOpacity:.85
      })
        // survol : aperçu rapide (nom + nombre d'éditions), pas de lien
        .bindTooltip('<b>' + nom + '</b><br>' + ville + ' · ' + n + ' édition(s)', {sticky:true, maxWidth:220})
        // clic : popup Leaflet, nativement interactive, avec le détail de chaque édition et
        // son lien "voir"
        .bindPopup('<b>' + nom + '</b><br>' + ville + ', ' + n + ' édition(s)<hr>' + listeEditions, {maxWidth:280});
    });
  }

  // Une carte Leaflet par ville, chacune cadrée sur son propre nuage (pas sur les 3 villes à
  // la fois : Lyon/Paris/Venise sont loin les unes des autres, une seule carte les englobant
  // montrerait surtout du vide entre elles). Fond OpenHistoricalMap (frontières et toponymes
  // d'époque), comme dans 01_carte_circulation.ipynb. Les deux jeux de marqueurs (éditeur et
  // graveur) sont construits à l'avance dans des L.layerGroup séparés : basculer ne fait que
  // retirer l'un et ajouter l'autre, sans reconstruire quoi que ce soit.
  const calquesParVille = {};
  for (const ville of Object.keys(villesCoords)) {
    const map = L.map('carte-' + ville, {scrollWheelZoom:false});
    L.maplibreGL({
      style: 'https://www.openhistoricalmap.org/map-styles/main/main.json',
      attribution: '© OpenHistoricalMap contributors'
    }).addTo(map);

    const editeursVille = pointsEditeurs.filter(p => p.ville === ville);
    const bornes = L.latLngBounds(editeursVille.map(p => [p.lat, p.lon]));
    map.fitBounds(bornes, {padding:[36, 36]});

    const calqueEditeur = L.layerGroup(construireMarqueurs(map, pointsEditeurs, ville, 'editeur'));
    const calqueGraveur = L.layerGroup(construireMarqueurs(map, pointsGraveurs, ville, 'graveur'));
    calqueEditeur.addTo(map);
    calquesParVille[ville] = {map, editeur: calqueEditeur, graveur: calqueGraveur};
  }

  // Bascule globale "Éditeur" / "Graveur" : change le calque affiché sur les 3 cartes.
  let modeGroupement = 'editeur';
  document.querySelectorAll('.bascule-groupement').forEach(bouton => {
    bouton.addEventListener('click', () => {
      const mode = bouton.dataset.mode;
      if (mode === modeGroupement) return;
      document.querySelectorAll('.bascule-groupement').forEach(b => b.classList.toggle('actif', b === bouton));
      for (const ville of Object.keys(calquesParVille)) {
        const c = calquesParVille[ville];
        c.map.removeLayer(c[modeGroupement]);
        c.map.addLayer(c[mode]);
      }
      modeGroupement = mode;
    });
  });

  // Un bouton "Afficher le tableau détaillé" par ville, juste sous sa carte.
  document.querySelectorAll('.action-ville').forEach(bouton => {
    bouton.addEventListener('click', () => {
      const tableau = document.getElementById('tableau-' + bouton.dataset.ville);
      const visible = tableau.classList.toggle('visible');
      bouton.textContent = visible ? 'Masquer le tableau détaillé' : 'Afficher le tableau détaillé';
    });
  });
</script>
</body></html>"""

html_final = (TEMPLATE_HTML
    .replace("__BLOCS_CARTE__", blocs_carte)
    .replace("__POINTS_EDITEURS__", json.dumps(points_editeurs, ensure_ascii=False))
    .replace("__POINTS_GRAVEURS__", json.dumps(points_graveurs, ensure_ascii=False))
    .replace("__VILLES_COORDS__", json.dumps(VILLES_COORDS, ensure_ascii=False)))

with open(CHEMIN_SORTIE, "w", encoding="utf-8") as f:
    f.write(html_final)

print("Carte écrite dans", CHEMIN_SORTIE)